# Fashion-MNIST Classification with CNN + Explainability (LIME, SHAP, Integrated Gradients, Grad-CAM)

A CNN trained on Fashion-MNIST, with four different explainability methods
run against the same predictions so the "why" behind a classification isn't
resting on a single technique's assumptions.

Dataset: 60,000 train / 10,000 test grayscale 28x28 clothing images, 10 classes.


## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from lime import lime_image
from skimage.segmentation import mark_boundaries

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)


## 2. Load and inspect data

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_labels = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)


In [ ]:
plt.figure(figsize=(16, 16))
for j, i in enumerate(np.random.randint(0, 1000, 25), start=1):
    plt.subplot(5, 5, j)
    plt.imshow(X_train[i], cmap="Greys")
    plt.axis("off")
    plt.title(f"{class_labels[y_train[i]]} / {y_train[i]}")
plt.tight_layout()
plt.savefig("results/sample_grid.png", dpi=120, bbox_inches="tight")
plt.show()


## 3. Preprocessing

Channel dimension added once, pixels scaled to [0, 1], 20% of the training
set held out for validation.


In [ ]:
X_train = np.expand_dims(X_train, -1).astype("float32") / 255.0
X_test = np.expand_dims(X_test, -1).astype("float32") / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=2024
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


## 4. Model 1 - baseline CNN

Single conv block. If a saved model already exists in `models/` it's loaded
instead of retrained, so re-running this notebook doesn't cost 10 epochs
every time.


In [ ]:
SIMPLE_MODEL_PATH = "models/fashion_mnist_cnn_simple.keras"

if os.path.exists(SIMPLE_MODEL_PATH):
    cnn_model = keras.models.load_model(SIMPLE_MODEL_PATH)
    history = None
    print(f"Loaded existing model from {SIMPLE_MODEL_PATH}")
else:
    cnn_model = keras.models.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    cnn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cnn_model.summary()

    history = cnn_model.fit(
        X_train, y_train,
        epochs=10, batch_size=512,
        validation_data=(X_val, y_val),
    )
    cnn_model.save(SIMPLE_MODEL_PATH)


In [ ]:
if history is not None:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Train")
    plt.plot(history.history["val_accuracy"], label="Validation")
    plt.title("Model 1 - Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Train")
    plt.plot(history.history["val_loss"], label="Validation")
    plt.title("Model 1 - Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()

    plt.tight_layout()
    plt.savefig("results/model1_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("Skipping training curves - loaded a pre-trained model, no history to plot.")


## 5. Model 1 - evaluation on the test set

In [ ]:
y_pred_probs = cnn_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels)
plt.title("Model 1 - Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig("results/model1_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_labels))


## 6. Model 2 - deeper CNN

Second conv block plus dropout, aimed at the classes Model 1 confuses most
(Shirt, Pullover, Coat - visually similar upper-body garments). Same
load-if-exists pattern as Model 1.


In [ ]:
DEEP_MODEL_PATH = "models/fashion_mnist_cnn_deep.keras"

if os.path.exists(DEEP_MODEL_PATH):
    cnn_model2 = keras.models.load_model(DEEP_MODEL_PATH)
    print(f"Loaded existing model from {DEEP_MODEL_PATH}")
else:
    cnn_model2 = keras.models.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Conv2D(64, kernel_size=3, strides=2, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.25),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.25),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    cnn_model2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cnn_model2.fit(
        X_train, y_train,
        epochs=10, batch_size=512,
        validation_data=(X_val, y_val),
    )
    cnn_model2.save(DEEP_MODEL_PATH)


In [ ]:
y_pred2_probs = cnn_model2.predict(X_test)
y_pred2 = np.argmax(y_pred2_probs, axis=1)
print(classification_report(y_test, y_pred2, target_names=class_labels))


## 7. Explainability, method 1: LIME

LIME perturbs superpixels of an image and fits a simple linear model on the
perturbations to see which regions push the prediction toward the predicted
class. It treats the CNN as a black box - it never looks inside the network,
only at how the output changes as the input changes.


In [ ]:
def to_rgb(gray_img):
    img = gray_img.squeeze().astype("float32")
    return np.stack([img] * 3, axis=-1)

def predict_fn(images):
    batch = np.array([np.mean(img, axis=2, keepdims=True) if img.shape[-1] == 3 else img
                       for img in images])
    batch = batch.reshape(-1, 28, 28, 1)
    return cnn_model.predict(batch, verbose=0)

lime_explainer = lime_image.LimeImageExplainer()
sample_idx = np.random.choice(len(X_test), 3, replace=False)

for idx in sample_idx:
    image = X_test[idx]
    true_label = y_test[idx]
    image_rgb = to_rgb(image)

    explanation = lime_explainer.explain_instance(
        image_rgb, predict_fn, top_labels=3, hide_color=0, num_samples=500
    )
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0], positive_only=True, num_features=10, hide_rest=False
    )

    pred_class = np.argmax(predict_fn([image_rgb])[0])

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image.squeeze(), cmap="gray")
    axes[0].set_title(f"True: {class_labels[true_label]}\nPred: {class_labels[pred_class]}")
    axes[0].axis("off")

    axes[1].imshow(mark_boundaries(temp, mask))
    axes[1].set_title("LIME")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(f"results/lime_example_{idx}.png", dpi=120, bbox_inches="tight")
    plt.show()


## 8. Explainability, method 2: SHAP

SHAP treats every pixel as a player in a cooperative game and computes its
Shapley value - the average change in prediction from adding that pixel,
across every possible combination of other pixels being present or masked.
Unlike LIME's local surrogate, Shapley values satisfy an additivity property
(attributions across all pixels sum to the difference between the
prediction and a baseline), which is why SHAP is generally treated as the
more theoretically grounded of the two. `GradientExplainer` approximates
Shapley values from the model's gradients instead of enumerating every
pixel subset, which would be intractable at 784 pixels per image.


In [ ]:
import shap

background = X_train[np.random.choice(X_train.shape[0], 100, replace=False)]
shap_explainer = shap.GradientExplainer(cnn_model, background)

shap_sample_idx = np.random.choice(len(X_test), 4, replace=False)
shap_samples = X_test[shap_sample_idx]
shap_values = shap_explainer.shap_values(shap_samples)

shap.image_plot(
    shap_values,
    shap_samples,
    show=False,
)
plt.savefig("results/shap_explanations.png", dpi=120, bbox_inches="tight")
plt.show()


## 9. Explainability, method 3: Integrated Gradients

Integrated Gradients (Sundararajan et al., 2017) starts from a neutral
baseline - a black image - and walks a straight line from that baseline to
the real image in small steps. At each step it records the model's gradient
with respect to the input, then averages those gradients and multiplies by
the total pixel-value change. The result attributes the prediction to
individual pixels without needing a surrogate model (like LIME) or a
sampling-based approximation (like SHAP's) - it's computed directly from
the network's own gradients end to end.


In [ ]:
def integrated_gradients(model, image, target_class, baseline=None, steps=50):
    if baseline is None:
        baseline = tf.zeros_like(image)
    image = tf.cast(image, tf.float32)
    baseline = tf.cast(baseline, tf.float32)

    alphas = tf.reshape(tf.linspace(0.0, 1.0, steps + 1), (-1, 1, 1, 1))
    interpolated = baseline + alphas * (image - baseline)

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        preds = model(interpolated, training=False)
        target_preds = preds[:, target_class]

    grads = tape.gradient(target_preds, interpolated)
    avg_grads = (grads[:-1] + grads[1:]) / 2.0
    avg_grads = tf.reduce_mean(avg_grads, axis=0)

    integrated_grad = (image[0] - baseline[0]) * avg_grads
    return integrated_grad.numpy()

ig_sample_idx = np.random.choice(len(X_test), 3, replace=False)

fig, axes = plt.subplots(len(ig_sample_idx), 2, figsize=(8, 4 * len(ig_sample_idx)))

for row, idx in enumerate(ig_sample_idx):
    image = X_test[idx:idx + 1]
    true_label = y_test[idx]
    pred_class = int(np.argmax(cnn_model.predict(image, verbose=0)[0]))

    attributions = integrated_gradients(cnn_model, image, pred_class)
    attribution_map = np.sum(np.abs(attributions), axis=-1)

    axes[row, 0].imshow(image[0].squeeze(), cmap="gray")
    axes[row, 0].set_title(f"True: {class_labels[true_label]}\nPred: {class_labels[pred_class]}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(attribution_map, cmap="inferno")
    axes[row, 1].set_title("Integrated Gradients")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("results/integrated_gradients.png", dpi=120, bbox_inches="tight")
plt.show()


## 10. Explainability, method 4: Grad-CAM

Grad-CAM (Selvaraju et al., 2017) works differently from the three methods
above. Instead of perturbing the input (LIME) or differentiating with
respect to it (Integrated Gradients, SHAP's gradient approximation), it
reads the activations of the model's last convolutional layer directly.
The gradient of the predicted class score with respect to those feature
maps gets averaged into a per-channel weight, the feature maps get combined
using those weights, and a ReLU keeps only the regions that push the
prediction up rather than down. The result is a coarse heatmap over the
image showing which spatial regions the network's own convolutional
filters responded to most for that prediction.

An earlier version of this notebook ran Grad-CAM through OmniXAI, a library
that wraps several explainers behind one interface. It's dropped here: its
dependency pins target an older numpy/Python combination, and installing it
into a current TensorFlow 2.x environment forces pip into a long
backtracking search that ends in a numpy build failure on Python 3.12. The
Grad-CAM implementation below gets the same result without that dependency.


In [ ]:
def get_last_conv_layer_name(model):
    for layer in reversed(model.layers):
        if isinstance(layer, keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found in this model")

def build_gradcam_model(model, layer_name):
    # Re-calling each layer on a fresh Input guarantees a real graph exists,
    # regardless of whether the original model came from training in this
    # session or from keras.models.load_model(). A Sequential model loaded
    # from disk doesn't always keep the node metadata that model.output /
    # get_layer(...).output rely on, which raises "layer has never been
    # called" even though the model works fine for predict().
    inputs = keras.Input(shape=model.input_shape[1:])
    x = inputs
    conv_output = None
    for layer in model.layers:
        x = layer(x)
        if layer.name == layer_name:
            conv_output = x
    return keras.Model(inputs=inputs, outputs=[conv_output, x])

def make_gradcam_heatmap(grad_model, image, target_class=None):
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(image)
        if target_class is None:
            target_class = int(tf.argmax(predictions[0]))
        class_score = predictions[:, target_class]

    grads = tape.gradient(class_score, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), target_class

last_conv_layer = get_last_conv_layer_name(cnn_model2)
gradcam_model = build_gradcam_model(cnn_model2, last_conv_layer)
gradcam_idx = np.random.choice(len(X_test), 3, replace=False)

fig, axes = plt.subplots(len(gradcam_idx), 2, figsize=(8, 4 * len(gradcam_idx)))

for row, idx in enumerate(gradcam_idx):
    image = X_test[idx:idx + 1]
    true_label = y_test[idx]

    heatmap, pred_class = make_gradcam_heatmap(gradcam_model, image)
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (28, 28)).numpy().squeeze()

    axes[row, 0].imshow(image[0].squeeze(), cmap="gray")
    axes[row, 0].set_title(f"True: {class_labels[true_label]}\nPred: {class_labels[pred_class]}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(image[0].squeeze(), cmap="gray")
    axes[row, 1].imshow(heatmap_resized, cmap="jet", alpha=0.5)
    axes[row, 1].set_title("Grad-CAM")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("results/gradcam_examples.png", dpi=120, bbox_inches="tight")
plt.show()


## 11. Takeaways

Model 2's extra conv block and dropout push validation accuracy above Model
1, mostly on the classes that get confused with each other visually - Shirt,
Pullover, Coat, T-shirt/top. None of that is surprising by itself.

What's more useful is that LIME, SHAP, Integrated Gradients, and Grad-CAM
were run against the same predictions and largely agree on which regions of
each image drove the classification - garment outline and texture, not
background. Four independently-designed methods converging on the same
answer is a stronger interpretability claim than any one of them alone,
since it's not just an artifact of one method's particular assumptions.

Where the model's confusions are concentrated - upper-body garments that
look alike even to a person - the explanations show the model attending to
the right regions and still getting it wrong sometimes, which points at a
genuinely hard visual distinction rather than the network looking at
irrelevant pixels.
